In [ ]:
from trustutils import run
run.TRUST_parameters()
run.introduction('MMS Exponential+Trig 2D', '2026')

run.reset()

NB_PROCS = 2

case = run.addCase('.', 'MMS_ExpTrig2D.data', nbProcs=NB_PROCS)
case.partition()          # trust -partition MMS_ExpTrig2D.data 2 -> PAR_MMS_ExpTrig2D.data + DOM_000*.Zones
# --- mesh-convergence scan (shared ladder: L1 = cells x2, L2 = cells x4) ---
# Levels instantiated from src/MMS_ExpTrig2D.tpl.data (byte-copy of the
# baseline dataset with $nx/$ny node-count placeholders); the baseline case
# above is the untouched L0. Each level runs in its own build/scan/<lvl>/
# directory so the partitioned DOM_000*.Zones never overwrite another's.
NX0 = NY0 = 21                        # baseline node counts (L0)
LEVELS = {
    "L1": {"nx": 2*NX0-1, "ny": 2*NY0-1},   # cells x2 -> h/2
    "L2": {"nx": 4*NX0-3, "ny": 4*NY0-3},   # cells x4 -> h/4
}
scan_cases = {}
for lvl, dic in LEVELS.items():
    c = run.addCaseFromTemplate("MMS_ExpTrig2D.tpl.data", f"scan/{lvl}", dic=dic,
                                targetData=f"MMS_ExpTrig2D_{lvl}.data",
                                nbProcs=NB_PROCS, excluNR=True)
    c.partition()          # trust -partition MMS_ExpTrig2D_L1.data 2 -> PAR_*.data + Zones
    scan_cases[lvl] = c
run.printCases()
run.runCases()
run.tablePerf()

In [ ]:
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# MMS_ExpTrig2D: b(x, y, 0) = sin(pi x) exp(-y), u = 0, eta = 0.01, Dirichlet
# zero at y = 0, 1 and Neumann at x = 0, 1.  The initial condition is not an
# eigenmode, but away from the thin wall layers the local decay rate is
# eta * |lap b| / b = eta * (pi^2 - 1); over the short nb_pas_dt_max-capped
# run the probe ratios must follow exp(-eta (pi^2 - 1) t) to a few percent.
eta = 0.01
rate = eta * (np.pi**2 - 1.0)

for name in ('SONDE_Y025', 'SONDE_Y050', 'SONDE_Y075'):
    t, b = read_son(case.dataFileName_, name)
    ratio = b[-1] / b[0]
    expected = np.exp(-rate * t[-1])
    err = abs(ratio - expected) / expected
    print(f"{name}: ratio={ratio:.4f} expected={expected:.4f} err={err:.3e}")
    assert err < 0.1, f"{name} decay error too large: {err:.3e}"
print("OK: exp-trig decay matches the local-rate prediction at all probes.")


## Mesh convergence

The shared refinement ladder: `L0` = the baseline mesh above (`21×21`),
`L1` = cells ×2 (`41×41`), `L2` = cells ×4 (`81×81`).

The *Checks* ratio metric saturates at the implicit-Euler temporal error,
which is common to every mesh. To expose the **spatial** error, the scan
compares $b(t)/b(0)$ against the exact implicit-Euler decay built with the
analytic local rate $\lambda = \eta(\pi^2 - 1)$ on the probe's own time
grid, $\mathrm{ref}(t_n) = \prod_j 1/(1+\lambda\,\Delta t_j)$: the time
scheme cancels out and the residual error is the spatial one. The observed
order is $\log_2(e_k/e_{k+1})$.

In [ ]:
SONDE_TAGS_SCAN = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
ETA = 0.01
LAM = ETA * (np.pi**2 - 1.0)   # analytic local decay rate eta*(pi^2 - 1)

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def implicit_ref(t, lam):
    """Exact implicit-Euler decay ratio on the probe time grid (any dt sequence)."""
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 / (1.0 + lam * (t[i] - t[i-1]))
    return ref

def spatial_ratio_error(subdir, case_name):
    """Max |b(t)/b(0) - implicit-Euler analytic decay| over the scan probes."""
    worst = 0.0
    for tag in SONDE_TAGS_SCAN:
        t, b = read_son_dir(subdir, case_name, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - implicit_ref(t, LAM)))))
    return worst

scan = {"L0": ("", case.dataFileName_),
        "L1": (f"scan/L1", scan_cases["L1"].dataFileName_),
        "L2": (f"scan/L2", scan_cases["L2"].dataFileName_)}
scan_nodes = {"L0": (21, 21), "L1": (41, 41), "L2": (81, 81)}
scan_err = {lvl: spatial_ratio_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes    |ratio - implicit ref|   observed order")
prev = None
for lvl in ("L0", "L1", "L2"):
    nx, ny = scan_nodes[lvl]
    e = scan_err[lvl]
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<6} {nx:>3}x{ny:<4} {e:<26.4e} {order}")
    prev = e

assert scan_err["L1"] < scan_err["L0"], f"no convergence L0 -> L1: {scan_err}"
assert scan_err["L2"] < scan_err["L1"], f"no convergence L1 -> L2: {scan_err}"
print("OK: spatial error decreases monotonically under refinement.")